# Recommend before you know the reader

A brand-new reader with no history just signed up. They have read nothing, rated nothing, searched for nothing. Yet the home page still needs books on it. **What do we recommend *before* we know anything about them?**

The honest first answer is *popularity*: recommend what everyone else reads. In this unit we build **popularity** as the first real retrieval path, measure it against Unit 1's random floor, and find that it genuinely wins. But winning a scoreboard is not the same as being right, so we also ask the uncomfortable question underneath it: **is the most-read book the best book?** Chasing that question honestly uncovers the sharpest idea in the unit -- that the offline metric is itself popularity-biased -- and hands us the motivation for everything that follows.

## Load the catalog and interactions

We reuse Unit 1's substrate exactly. `bookrec.generated_dir()` locates the seeded data slice (run the recsys generators first if it complains), `load_catalog` returns the `dict[int, Book]` keyed by stable `item_id`, and the interaction log is a gzip-ed CSV with one row per reader-item event.

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
interactions = pd.read_csv(data_dir / "interactions.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
len(catalog), len(interactions)

## 1. Count popularity -- on the training split only

The simplest signal that reads data is a **count**: how many readers liked each book? "Liked" means a positive interaction (`label == 1`). Popularity ranks items by that count and recommends the top of the list to everyone -- it is *reader-independent*, the same ranking for every query. That is exactly what we want for a brand-new reader with no history.

There is one rule we must not break. The count is computed on the **train** split *only*. The `val` split is the sealed holdout we score against, and `test` is sealed tighter still. If we counted popularity over `val`, the ranker would have peeked at the very interactions we are about to grade it on -- **leakage** -- and every number afterwards would be a lie. Train to learn, val to measure.

```text
  interactions
       |
       |  keep split == "train" AND label == 1   (never val / test)
       v
  count positives per item_id
       |
       v
  rank items by count  -->  top-k is the SAME list for every reader
```

### Count by hand, then reveal the library

First the mechanism, in pandas: filter to positive train rows, group by item, and count.

In [ ]:
train = interactions[interactions["split"] == "train"]
train_positives = train[train["label"] == 1]
counts_by_hand = train_positives.groupby("item_id").size().sort_values(ascending=False)
counts_by_hand.head()

### The distribution is heavy-tailed

Popularity is not spread evenly. A handful of books collect many positives while most collect one or two -- and more than a thousand catalog items were *never* a train positive at all. This long tail is the whole story of the unit: a count ranker can only ever recommend from the short, crowded head.

In [ ]:
ever = len(counts_by_hand)
never = len(catalog) - ever
singletons = int((counts_by_hand == 1).sum())
tail_fraction = float((counts_by_hand <= 2).mean())

print(f"{ever} of {len(catalog)} books were ever a positive on train; {never} never were")
print(f"{singletons} of those appear exactly once; {tail_fraction:.0%} appear at most twice")
print(f"most-read book: {int(counts_by_hand.iloc[0])} positives")
print()
print("top of the distribution (one '#' is a few positives):")
scale = int(counts_by_hand.iloc[0])
for item_id, count in zip(counts_by_hand.index[:10], counts_by_hand.values[:10]):
    bar = "#" * max(1, round(40 * count / scale))
    print(f"  item {item_id:5d} | {bar} {count}")

### Reveal `PopularityRetrievalPath`

The package ships this exact count as a retrieval path. `fit` takes an iterable of row mappings with keys `reader_id, item_id, split, label` -- the shape a pandas `DataFrame.to_dict("records")` produces -- counts only positive train rows (leakage safety lives *in the code*, not in our discipline), and `retrieve` returns the top items by count, honouring `context["seen"]` so a book already read is never recommended back.

In [ ]:
rows = interactions.to_dict("records")
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)

# the shipped counts match our by-hand count exactly
by_hand = {int(i): int(c) for i, c in zip(counts_by_hand.index, counts_by_hand.values)}
assert popularity.counts == by_hand

top5 = popularity.retrieve(query="any-reader", context={"seen": set()}, k=5)
[(c.item_id, int(c.score)) for c in top5]

### Does it beat the random floor?

Unit 1 gave us a scoreboard: `run_validation_scoreboard` scores a path on the positive `val` rows, removes each reader's already-seen train items from both the recommendations and the relevance set, and averages hit-rate@k over the eligible readers. The random floor is the easiest thing to beat.

We pass `cold_readers` from `cold_partitions.json` as a matter of hygiene -- cold readers must never be scored. (On *this* seed the cold readers happen to have no positive `val` rows, so the exclusion changes nothing here: 368 readers are scored either way. It is still the correct call to make, because a future data slice will have cold readers with val activity.)

In [ ]:
cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])

random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=0)

pop_board = bookrec.run_validation_scoreboard(
    popularity, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
)
random_board = bookrec.run_validation_scoreboard(
    random_floor, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
)

print(f"readers scored:   {pop_board.readers}")
print(f"random hit@10:    {random_board.hit_rate_at_k:.4f}")
print(f"popularity hit@10:{pop_board.hit_rate_at_k:.4f}")
print(f"ratio:            {pop_board.hit_rate_at_k / random_board.hit_rate_at_k:.1f}x the floor")

Popularity scores about **0.12** against a random floor of about **0.014** -- roughly **9x** the floor, across 368 readers. The simplest path that reads data really does work: before we know anything about a reader, recommending what the crowd reads is a defensible first guess. That is why popularity is the *cold-start* fallback in almost every real system.

So we have a winner. Now we press on it.

## 2. Is the most-read the best?

Count popularity answers "what is read most", not "what is liked most". A book can be read 10000 times and liked 9000 of them (a 90% positive *rate*), or read 3 times and liked all 3 (a 100% rate on almost no evidence). Ranked by raw rate, the 3-of-3 book wins -- which is absurd. Three data points cannot outvote ten thousand.

The fix is a **weighted (Bayesian-shrinkage) rating**. We blend each item's own rate with the global average, weighting by how much evidence the item has:

$$\text{score} = \frac{v\,R + m\,C}{v + m}$$

where for one item $R = \tfrac{\text{positives}}{\text{exposures}}$ is its own positive rate, $v$ is its exposure (how many times it was shown/read on train), $C$ is the global positive rate across all items, and $m$ is a prior strength measured in *pseudo-counts* -- think of it as "$m$ phantom observations pinned at the global average $C$, mixed in before we trust the item's own $v$ observations". (An advanced reader may recognise this as the same "weighted rating" IMDb publishes for its Top 250 chart -- we derive it here from scratch as Bayesian shrinkage, so the name is just a place-marker for a formula we already understand.)

### Read the formula at its limits

Rewrite the score as a pull away from the global average:

$$\text{score} = C + \frac{v\,(R - C)}{v + m}$$

- **$m \to 0$**: the phantom counts vanish and $\text{score} \to R$ -- the raw rate, no shrinkage. The 3-of-3 book is back to a perfect 1.0.
- **growing $m$**: the fraction $\frac{v}{v+m}$ shrinks, so every score **contracts toward $C$**. An item with thin evidence (small $v$) is pulled hard toward the average; an item with deep evidence (large $v$) barely moves. That is exactly the behaviour we wanted -- $m$ is the price in evidence an item must pay before we believe its rate.

One honest caveat about large $m$. The scores all crowd toward $C$, but the *ranking* does **not** collapse to a flat tie at any finite $m$: since the gap is $|v\,(R-C)|/(v+m)$, for large $m$ the order approaches ranking by $v\,(R-C)$ -- an exposure-weighted "how far above average". Exact all-equal ties appear only in the $m \to \infty$ limit. Shrinkage re-weights evidence; it does not erase it.

```text
raw rate R (m->0)      thin item o                      o deep item
                                  \                    /
          shrink toward C  ........\........m.........././.......... C (global)
                                    \                /
large m                              o (pulled far)  o (barely moves)
```

In [ ]:
# exposures and positives per catalog item, on TRAIN only
exposure_by = train.groupby("item_id").size()
positive_by = train_positives.groupby("item_id").size()
exposure_map = {int(i): int(v) for i, v in zip(exposure_by.index, exposure_by.values)}
positive_map = {int(i): int(v) for i, v in zip(positive_by.index, positive_by.values)}

exposures = np.array([exposure_map[i] if i in exposure_map else 0 for i in catalog_ids])
positives = np.array([positive_map[i] if i in positive_map else 0 for i in catalog_ids])

C = float(positives.sum() / exposures.sum())
print(f"global positive rate C = {C:.4f}  ({int(positives.sum())} positives over {int(exposures.sum())} train exposures)")

### Reveal `weighted_rating`, and watch the thin book lose

`bookrec.weighted_rating(positives, exposures, m=..., global_rate=...)` computes the score vectorised over item arrays. It is deliberately *id-free* and does no tie-breaking -- it returns one number per item and leaves ordering to the ranker. Here is the 3-of-3 vs 9000-of-10000 showdown at `m = 10`.

In [ ]:
demo_pos = np.array([3, 9000])
demo_exp = np.array([3, 10000])

raw = bookrec.weighted_rating(demo_pos, demo_exp, m=0.0, global_rate=C)        # m->0 recovers R
shrunk = bookrec.weighted_rating(demo_pos, demo_exp, m=10.0, global_rate=C)    # m=10 pseudo-counts

print(f"raw rate R (m->0):   3/3 = {raw[0]:.3f}   9000/10000 = {raw[1]:.3f}   (thin book 'wins')")
print(f"weighted (m=10):     3/3 = {shrunk[0]:.3f}   9000/10000 = {shrunk[1]:.3f}   (thin book loses)")

### Re-rank the whole catalog by quality

Apply the same weighted rating to every catalog item and compare the top of the *count* ranking with the top of the *quality* ranking. Different books surface: the count list is full of heavily-exposed items near the global rate, while the quality list promotes lightly-exposed items whose rate sits above average.

In [ ]:
quality_scores = bookrec.weighted_rating(positives, exposures, m=10.0, global_rate=C)
quality_of = {int(i): float(s) for i, s in zip(catalog_ids, quality_scores)}
count_of = dict(popularity.counts)

def rate_of(item_id):
    v = exposure_map[item_id] if item_id in exposure_map else 0
    p = positive_map[item_id] if item_id in positive_map else 0
    return p / v if v else 0.0

top_count = sorted(catalog_ids, key=lambda i: (-(count_of[i] if i in count_of else 0), i))[:5]
top_quality = sorted(catalog_ids, key=lambda i: (-quality_of[i], i))[:5]

print("TOP 5 BY COUNT (popularity)")
print(f"  {'item':>6} {'count':>6} {'expo':>6} {'rate':>6} {'quality':>8}")
for i in top_count:
    expo = exposure_map[i] if i in exposure_map else 0
    print(f"  {i:>6} {count_of[i]:>6} {expo:>6} {rate_of(i):>6.2f} {quality_of[i]:>8.3f}")

print("\nTOP 5 BY QUALITY (weighted rating)")
print(f"  {'item':>6} {'count':>6} {'expo':>6} {'rate':>6} {'quality':>8}")
for i in top_quality:
    expo = exposure_map[i] if i in exposure_map else 0
    cnt = count_of[i] if i in count_of else 0
    print(f"  {i:>6} {cnt:>6} {expo:>6} {rate_of(i):>6.2f} {quality_of[i]:>8.3f}")

### Now score the quality ranking honestly

Quality looks like the better idea -- it promotes books readers actually *liked* rather than books merely *shown*. So let us hold it to the same scoreboard. We build a tiny reader-independent path that ranks by the quality score (structurally identical to the popularity path, just a different score) and measure it on `val`.

In [ ]:
class QualityRetrievalPath:
    """Reader-independent ranking by weighted-rating quality score (the count path's twin)."""

    name = "quality"

    def __init__(self, item_ids, scores):
        self._pairs = [(int(i), float(s)) for i, s in zip(item_ids, scores)]

    def retrieve(self, query, context, k):
        seen = set(context["seen"]) if "seen" in context else set()
        candidates = [
            bookrec.Candidate(item_id, score, "quality")
            for item_id, score in self._pairs
            if item_id not in seen
        ]
        return bookrec.rank(candidates, n=k)


quality_path = QualityRetrievalPath(catalog_ids, quality_scores)
quality_board = bookrec.run_validation_scoreboard(
    quality_path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
)

print(f"popularity (count) hit@10: {pop_board.hit_rate_at_k:.4f}")
print(f"quality (weighted) hit@10: {quality_board.hit_rate_at_k:.4f}")

### The sharpest idea in the unit: the metric is popularity-biased

The quality ranking scores **near zero** -- about **0.005**, which lands *below* even the random floor (about **0.014**) and far below the count path. The "better" idea did not just lose; it finished last. Why?

Ranking by quality (positive *rate*) promotes **lightly-exposed** items whose rate sits above average -- but those are exactly the items the holdout almost never contains. Look at how `val` was built: a reader's `val` positives are books they actually encountered, and **exposure is driven by popularity**, so the held-out positives are overwhelmingly the already-popular ones. A quality-first ranker therefore spends its slots on books that are *anti-correlated* with the holdout -- the genuinely-good-but-barely-seen items almost no one was given the chance to read and log in `val` -- so hit@k scores it near zero. The same metric rewards recommending the already-popular and *punishes* surfacing a good but lightly-exposed book.

This is **popularity bias living inside the metric itself**. The quality ranker is not broken; the measuring stick is bent. Two consequences follow, and both shape the rest of the book:

- An offline win is necessary but not sufficient. A path can win hit@k by pandering to popularity while serving the catalog poorly.
- We cannot fully trust accuracy alone. We need to *measure* the bias directly -- which is exactly the next section.

## 3. Measure popularity bias directly

If accuracy can be won by pandering to the head, we should measure how much of the catalog a recommender actually *uses*. Two beyond-accuracy metrics make the bias visible. Both are computed over the **same scored-reader set** the scoreboard used (cold readers already excluded), with the head defined on **train** exposure.

- **head set** -- the top 10% of catalog items by train positive count (`head_ids_from_counts`), ties broken by ascending item id, so exactly `ceil(0.10 * N)` items.
- **catalog coverage** -- the fraction of *unique* catalog items that appear in at least one reader's top-k. High coverage means the system spreads attention across the catalog.
- **head-share** -- the fraction of recommendation *slots* (summed over every scored reader's top-k) whose item is in the head set. High head-share means the system keeps serving the same crowded few.

To compute them we need each scored reader's actual top-k list, so we reproduce the scoreboard's eligibility rule (a non-cold reader with at least one unseen positive `val` item) and collect the recommendations.

In [ ]:
def scored_recommendations(path):
    """Reproduce the scoreboard's scored-reader set and collect each one's top-k list."""
    catalog_set = set(catalog_ids)
    seen_by = defaultdict(set)
    val_by = defaultdict(set)
    for row in rows:
        if row["item_id"] not in catalog_set or row["label"] != 1:
            continue
        if row["split"] == "train":
            seen_by[row["reader_id"]].add(row["item_id"])
        elif row["split"] == "val":
            val_by[row["reader_id"]].add(row["item_id"])
    recommendations = []
    for reader_id in sorted(val_by):
        seen = seen_by[reader_id]
        if reader_id in cold_readers or not (val_by[reader_id] - seen):
            continue
        candidates = path.retrieve(reader_id, {"seen": seen}, 10)
        recommendations.append(bookrec.rank(candidates, n=10, exclude=seen))
    return recommendations


head = bookrec.head_ids_from_counts(popularity.counts, catalog_ids, fraction=0.10)
pop_recs = scored_recommendations(popularity)
random_recs = scored_recommendations(random_floor)

print(f"head set size: {len(head)}  (top 10% of {len(catalog_ids)} items)")
print()
print(f"{'path':>12} {'coverage':>10} {'head-share':>11}")
for label, recs in (("popularity", pop_recs), ("random", random_recs)):
    cov = bookrec.catalog_coverage(recs, catalog_ids)
    hs = bookrec.head_share(recs, head)
    print(f"{label:>12} {cov:>10.4f} {hs:>11.4f}")

### Name the bias

The numbers are stark. The popularity "winner" has a catalog coverage of about **0.007** -- it serves roughly *14 of 2000* books, ever, to anyone -- and a head-share of **1.0**: every single recommendation slot is a head item. The random floor, which we just beat on accuracy, has coverage about **0.84** and head-share about **0.10**. The embarrassing baseline is vastly *fairer* to the catalog than our scoreboard champion.

This is **popularity bias**: a recommender that concentrates exposure on a few head items and starves the long tail. Left unchecked it is self-reinforcing -- recommending popular books makes them more popular, which makes them more recommended. It is a real cost, not a rounding error: the reader never discovers the tail, and the authors in the tail never get discovered.

```text
coverage:   popularity |#                                   | ~0.007
            random     |##################################  | ~0.84

head-share: popularity |##################################  | 1.0
            random     |###                                 | ~0.10
```

### Where this goes

We now have the two sides of the first real recommender and the vocabulary to judge it honestly:

- Popularity is a strong, defensible **cold-start** path -- it beats the floor and needs no reader history. Keep it.
- But it is biased toward the head, and the offline metric quietly rewards that bias. Accuracy alone will mislead us.

The way out of "recommend the same books to everyone" is **personalisation** -- using *this* reader's history to retrieve books suited to them -- which arrives with neighbourhood collaborative filtering in **Unit 4**. And the way to keep accuracy honest is to track coverage, head-share, and the rest of the **beyond-accuracy** metrics alongside hit@k from here on, deepened in **Unit 13**. Popularity is where every recommender starts; it is not where a good one stops.